# Stage 2: Batch Transformation Pipeline

Project Topic: SupplyChain Manufacturing

This notebook implements the Spark batch transformation pipeline for the PrecisionParts manufacturing scenario. Stage 2 reads raw data from the HDFS landing zone, cleans and standardizes each dataset, joins data sources to build enriched views, and aggregates results to answer PrecisionParts' core business questions.

The five datasets processed here are:
- `production-lines.csv.gz` — shift-level production metrics per factory and line
- `inventory-levels.csv.gz` — daily stock levels, reorder points, and lead times per product
- `equipment-sensors.csv.gz` — time-series sensor readings (temperature, vibration, pressure) per machine
- `quality-inspections.json.gz` — inspection results with defect codes and severity per batch
- `supplier-performance.json.gz` — on-time delivery rate, defect rate, and lead time per supplier

All curated outputs are written as Parquet with appropriate partitioning. All analytics outputs are written as Parquet and answer specific PrecisionParts business questions.

## 1. SparkSession Initialization

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, DateType
)
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("SupplyChain-Stage2-BatchTransforms") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.executor.memory", "1g") \
    .config("spark.driver.memory", "1g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print("SparkSession initialized successfully.")

## 2. Path Configuration

All paths point to HDFS. The landing zone holds the original compressed files loaded in Stage 1. Curated and analytics outputs are written back to HDFS.

In [ ]:
BASE = "hdfs://namenode:9000/data/supplychain"

LANDING = {
    "production":  f"{BASE}/landing/production_lines/production-lines.csv.gz",
    "inventory":   f"{BASE}/landing/inventory_levels/inventory-levels.csv.gz",
    "sensors":     f"{BASE}/landing/equipment_sensors/equipment-sensors.csv.gz",
    "quality":     f"{BASE}/landing/quality_inspections/quality-inspections.json.gz",
    "supplier":    f"{BASE}/landing/supplier_performance/supplier-performance.json.gz",
}

CURATED = {
    "production":  f"{BASE}/curated/production_lines",
    "inventory":   f"{BASE}/curated/inventory_levels",
    "sensors":     f"{BASE}/curated/equipment_sensors",
    "quality":     f"{BASE}/curated/quality_inspections",
    "supplier":    f"{BASE}/curated/supplier_performance",
}

ANALYTICS = {
    "defect_analysis":     f"{BASE}/analytics/defect_analysis",
    "inventory_risk":      f"{BASE}/analytics/inventory_risk",
    "supplier_scorecard":  f"{BASE}/analytics/supplier_scorecard",
    "equipment_monitoring":f"{BASE}/analytics/equipment_monitoring",
}

print("Path configuration set.")
for zone, paths in [("LANDING", LANDING), ("CURATED", CURATED), ("ANALYTICS", ANALYTICS)]:
    print(f"\n{zone} zone:")
    for k, v in paths.items():
        print(f"  {k}: {v}")

## 3. Read Raw Data from Landing Zone

Spark reads gzipped CSV and JSON files natively — no manual decompression needed. `inferSchema=True` is used here during exploration; explicit schemas are applied during cleaning.

In [ ]:
# --- Production Lines ---
raw_production = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(LANDING["production"])

print("production-lines schema:")
raw_production.printSchema()
print(f"Row count: {raw_production.count():,}")
raw_production.show(5, truncate=False)

In [ ]:
# --- Inventory Levels ---
raw_inventory = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(LANDING["inventory"])

print("inventory-levels schema:")
raw_inventory.printSchema()
print(f"Row count: {raw_inventory.count():,}")
raw_inventory.show(5, truncate=False)

In [ ]:
# --- Equipment Sensors ---
raw_sensors = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(LANDING["sensors"])

print("equipment-sensors schema:")
raw_sensors.printSchema()
print(f"Row count: {raw_sensors.count():,}")
raw_sensors.show(5, truncate=False)

In [ ]:
# --- Quality Inspections ---
raw_quality = spark.read \
    .option("multiline", "true") \
    .json(LANDING["quality"])

print("quality-inspections schema:")
raw_quality.printSchema()
print(f"Row count: {raw_quality.count():,}")
raw_quality.show(5, truncate=False)

In [ ]:
# --- Supplier Performance ---
raw_supplier = spark.read \
    .option("multiline", "true") \
    .json(LANDING["supplier"])

print("supplier-performance schema:")
raw_supplier.printSchema()
print(f"Row count: {raw_supplier.count():,}")
raw_supplier.show(5, truncate=False)

## 4. Data Cleaning and Standardization

Each dataset is cleaned independently before joining. Cleaning steps applied:
- Drop duplicate rows
- Drop rows where primary key or critical fields are null
- Cast columns to correct types (timestamps, dates, numerics)
- Standardize string fields (trim whitespace, lowercase where appropriate)
- Filter out physically impossible sensor values
- Normalize rate columns (e.g., on_time_delivery_rate) to [0, 1] if stored as percentages

In [ ]:
# ---- Clean: Production Lines ----

clean_production = raw_production \
    .dropDuplicates() \
    .dropna(subset=["production_line_id", "factory_id", "shift_date"]) \
    .withColumn("shift_date", F.to_date(F.col("shift_date"))) \
    .withColumn("units_produced",  F.col("units_produced").cast(IntegerType())) \
    .withColumn("units_defective", F.col("units_defective").cast(IntegerType())) \
    .withColumn("downtime_minutes",F.col("downtime_minutes").cast(DoubleType())) \
    .withColumn("factory_id",      F.trim(F.col("factory_id"))) \
    .withColumn("shift",           F.trim(F.lower(F.col("shift")))) \
    .filter(F.col("units_produced") >= 0) \
    .filter(F.col("units_defective") >= 0) \
    .filter(F.col("downtime_minutes") >= 0)

print(f"Production rows after cleaning: {clean_production.count():,}")
clean_production.show(5, truncate=False)

In [ ]:
# ---- Clean: Inventory Levels ----

clean_inventory = raw_inventory \
    .dropDuplicates() \
    .dropna(subset=["product_id", "snapshot_date"]) \
    .withColumn("snapshot_date",  F.to_date(F.col("snapshot_date"))) \
    .withColumn("stock_level",    F.col("stock_level").cast(IntegerType())) \
    .withColumn("reorder_point",  F.col("reorder_point").cast(IntegerType())) \
    .withColumn("lead_time_days", F.col("lead_time_days").cast(IntegerType())) \
    .withColumn("product_id",     F.trim(F.col("product_id"))) \
    .filter(F.col("stock_level") >= 0) \
    .filter(F.col("reorder_point") >= 0)

print(f"Inventory rows after cleaning: {clean_inventory.count():,}")
clean_inventory.show(5, truncate=False)

In [ ]:
# ---- Clean: Equipment Sensors ----
# Filter out readings outside physically plausible ranges.
# These thresholds are illustrative — adjust to match the actual dataset ranges.

clean_sensors = raw_sensors \
    .dropDuplicates() \
    .dropna(subset=["machine_id", "factory_id", "reading_timestamp"]) \
    .withColumn("reading_timestamp", F.to_timestamp(F.col("reading_timestamp"))) \
    .withColumn("reading_date",      F.to_date(F.col("reading_timestamp"))) \
    .withColumn("temperature_c",     F.col("temperature_c").cast(DoubleType())) \
    .withColumn("vibration_mm_s",    F.col("vibration_mm_s").cast(DoubleType())) \
    .withColumn("pressure_bar",      F.col("pressure_bar").cast(DoubleType())) \
    .withColumn("factory_id",        F.trim(F.col("factory_id"))) \
    .withColumn("machine_id",        F.trim(F.col("machine_id"))) \
    .filter(F.col("temperature_c").between(-20, 500)) \
    .filter(F.col("vibration_mm_s") >= 0) \
    .filter(F.col("pressure_bar") >= 0)

print(f"Sensor rows after cleaning: {clean_sensors.count():,}")
clean_sensors.show(5, truncate=False)

In [ ]:
# ---- Clean: Quality Inspections ----

clean_quality = raw_quality \
    .dropDuplicates() \
    .dropna(subset=["inspection_id", "product_id", "inspection_date"]) \
    .withColumn("inspection_date",   F.to_date(F.col("inspection_date"))) \
    .withColumn("defect_count",      F.col("defect_count").cast(IntegerType())) \
    .withColumn("product_id",        F.trim(F.col("product_id"))) \
    .withColumn("defect_code",       F.trim(F.lower(F.col("defect_code")))) \
    .withColumn("severity",          F.trim(F.lower(F.col("severity")))) \
    .filter(F.col("defect_count") >= 0)

print(f"Quality inspection rows after cleaning: {clean_quality.count():,}")
clean_quality.show(5, truncate=False)

In [ ]:
# ---- Clean: Supplier Performance ----
# Normalize rate fields: if stored as 0–100, divide by 100 to get 0.0–1.0

clean_supplier = raw_supplier \
    .dropDuplicates() \
    .dropna(subset=["supplier_id", "report_month"]) \
    .withColumn("report_month",            F.to_date(F.col("report_month"), "yyyy-MM")) \
    .withColumn("on_time_delivery_rate",   F.col("on_time_delivery_rate").cast(DoubleType())) \
    .withColumn("supplier_defect_rate",    F.col("supplier_defect_rate").cast(DoubleType())) \
    .withColumn("avg_lead_time_days",      F.col("avg_lead_time_days").cast(DoubleType())) \
    .withColumn("supplier_id",             F.trim(F.col("supplier_id"))) \
    .withColumn("country",                 F.trim(F.col("country"))) \
    .withColumn("on_time_delivery_rate",
        F.when(F.col("on_time_delivery_rate") > 1.0,
               F.col("on_time_delivery_rate") / 100.0)
         .otherwise(F.col("on_time_delivery_rate"))) \
    .withColumn("supplier_defect_rate",
        F.when(F.col("supplier_defect_rate") > 1.0,
               F.col("supplier_defect_rate") / 100.0)
         .otherwise(F.col("supplier_defect_rate"))) \
    .filter(F.col("on_time_delivery_rate").between(0, 1)) \
    .filter(F.col("supplier_defect_rate").between(0, 1))

print(f"Supplier rows after cleaning: {clean_supplier.count():,}")
clean_supplier.show(5, truncate=False)

## 5. Write Curated Zone (Parquet with Partitioning)

Each cleaned dataset is written to the HDFS curated zone as Parquet. Partition keys follow the strategy defined in Stage 1.

In [ ]:
# Production: partition by factory_id and shift_date
clean_production.write \
    .mode("overwrite") \
    .partitionBy("factory_id", "shift_date") \
    .parquet(CURATED["production"])
print("Curated production_lines written.")

# Inventory: partition by snapshot_date
clean_inventory.write \
    .mode("overwrite") \
    .partitionBy("snapshot_date") \
    .parquet(CURATED["inventory"])
print("Curated inventory_levels written.")

# Sensors: partition by factory_id and reading_date
clean_sensors.write \
    .mode("overwrite") \
    .partitionBy("factory_id", "reading_date") \
    .parquet(CURATED["sensors"])
print("Curated equipment_sensors written.")

# Quality: partition by product_id
clean_quality.write \
    .mode("overwrite") \
    .partitionBy("product_id") \
    .parquet(CURATED["quality"])
print("Curated quality_inspections written.")

# Supplier: partition by country
clean_supplier.write \
    .mode("overwrite") \
    .partitionBy("country") \
    .parquet(CURATED["supplier"])
print("Curated supplier_performance written.")

## 6. Read Back Curated Data

We read from the curated Parquet files for all downstream joins and aggregations. This ensures joins operate on clean, typed data.

In [ ]:
production = spark.read.parquet(CURATED["production"])
inventory  = spark.read.parquet(CURATED["inventory"])
sensors    = spark.read.parquet(CURATED["sensors"])
quality    = spark.read.parquet(CURATED["quality"])
supplier   = spark.read.parquet(CURATED["supplier"])

print("Curated datasets loaded:")
for name, df in [("production", production), ("inventory", inventory),
                 ("sensors", sensors), ("quality", quality), ("supplier", supplier)]:
    print(f"  {name}: {df.count():,} rows")

## 7. Joins and Enriched Datasets

### Join 1: Production + Quality Inspections

This join links shift-level production data to batch-level quality inspection outcomes. The join key is `product_id` — each production shift produces products that appear in quality inspections. This enriched dataset is the foundation for defect root-cause analysis, linking factory, shift, and production volume to defect outcomes.

In [ ]:
# Join 1: Production + Quality (left join — keep all production records,
# even if a shift has no corresponding inspection yet)
production_quality = production.join(
    quality,
    on="product_id",
    how="left"
).select(
    production["production_line_id"],
    production["factory_id"],
    production["shift_date"],
    production["shift"],
    production["product_id"],
    production["units_produced"],
    production["units_defective"],
    production["downtime_minutes"],
    quality["inspection_id"],
    quality["defect_count"],
    quality["defect_code"],
    quality["severity"],
    quality["inspection_date"]
)

print(f"Production + Quality join rows: {production_quality.count():,}")
production_quality.show(5, truncate=False)

### Join 2: Production + Supplier Performance

This join links production-line defect data to the supplier who provided materials for each production line. The join key is `supplier_id`. This is essential for answering whether high supplier defect rates correlate with downstream production defects.

In [ ]:
# Join 2: Production + Supplier (inner join — only analyze production
# records where a supplier can be identified)
production_supplier = production.join(
    supplier,
    on="supplier_id",
    how="inner"
).select(
    production["production_line_id"],
    production["factory_id"],
    production["shift_date"],
    production["product_id"],
    production["units_produced"],
    production["units_defective"],
    supplier["supplier_id"],
    supplier["supplier_name"],
    supplier["country"],
    supplier["on_time_delivery_rate"],
    supplier["supplier_defect_rate"],
    supplier["avg_lead_time_days"]
)

print(f"Production + Supplier join rows: {production_supplier.count():,}")
production_supplier.show(5, truncate=False)

### Join 3: Equipment Sensors + Production

This join connects machine sensor readings to the production line they belong to, enabling analysis of whether abnormal sensor readings (high temperature, vibration spikes) are associated with elevated defect rates or downtime.

In [ ]:
# Join 3: Sensors + Production (join on factory_id and date)
# Aggregate daily sensor statistics per factory before joining
daily_sensor_stats = sensors.groupBy("factory_id", "reading_date").agg(
    F.avg("temperature_c").alias("avg_temp_c"),
    F.max("temperature_c").alias("max_temp_c"),
    F.avg("vibration_mm_s").alias("avg_vibration"),
    F.max("vibration_mm_s").alias("max_vibration"),
    F.avg("pressure_bar").alias("avg_pressure"),
    F.count("machine_id").alias("sensor_readings_count")
)

sensors_production = daily_sensor_stats.join(
    production,
    on=[
        daily_sensor_stats["factory_id"] == production["factory_id"],
        daily_sensor_stats["reading_date"] == production["shift_date"]
    ],
    how="inner"
).select(
    daily_sensor_stats["factory_id"],
    daily_sensor_stats["reading_date"],
    production["production_line_id"],
    production["shift"],
    production["units_produced"],
    production["units_defective"],
    production["downtime_minutes"],
    "avg_temp_c", "max_temp_c",
    "avg_vibration", "max_vibration",
    "avg_pressure", "sensor_readings_count"
)

print(f"Sensors + Production join rows: {sensors_production.count():,}")
sensors_production.show(5, truncate=False)

## 8. Analytics Aggregations

The following aggregations directly answer PrecisionParts' business questions. Each output is written to the analytics zone as Parquet.

### Analytics 1: Defect Rate by Factory, Line, and Shift

**Business question:** Which factories, production lines, and shifts have the highest defect rates? This drives targeted intervention and shift-scheduling decisions.

In [ ]:
defect_by_line_shift = production.groupBy(
    "factory_id", "production_line_id", "shift"
).agg(
    F.sum("units_produced").alias("total_units_produced"),
    F.sum("units_defective").alias("total_units_defective"),
    F.sum("downtime_minutes").alias("total_downtime_minutes"),
    F.count("shift_date").alias("shift_count"),
    F.round(
        F.sum("units_defective") / F.sum("units_produced") * 100, 2
    ).alias("defect_rate_pct")
).orderBy(F.desc("defect_rate_pct"))

print("Defect rate by factory / line / shift:")
defect_by_line_shift.show(20, truncate=False)

defect_by_line_shift.write \
    .mode("overwrite") \
    .parquet(f"{ANALYTICS['defect_analysis']}/defect_by_line_shift")
print("Written: defect_by_line_shift")

### Analytics 2: Top Defect Codes by Frequency and Severity

**Business question:** What are the most common defect types, and how severe are they? This helps quality engineers prioritize which defect categories to address first.

In [ ]:
defect_code_summary = quality.groupBy("defect_code", "severity").agg(
    F.count("inspection_id").alias("inspection_count"),
    F.sum("defect_count").alias("total_defects"),
    F.avg("defect_count").alias("avg_defects_per_inspection")
).orderBy(F.desc("total_defects"))

print("Top defect codes by frequency and severity:")
defect_code_summary.show(20, truncate=False)

defect_code_summary.write \
    .mode("overwrite") \
    .parquet(f"{ANALYTICS['defect_analysis']}/defect_code_summary")
print("Written: defect_code_summary")

### Analytics 3: Inventory Risk — Products Below Reorder Point

**Business question:** Which products are at risk of stockout right now, and how many days of stock remain? This directly supports inventory replenishment decisions.

In [ ]:
# Get the most recent snapshot for each product
latest_snapshot_window = Window.partitionBy("product_id").orderBy(F.desc("snapshot_date"))

latest_inventory = inventory \
    .withColumn("rank", F.rank().over(latest_snapshot_window)) \
    .filter(F.col("rank") == 1) \
    .drop("rank")

inventory_risk = latest_inventory \
    .withColumn("stock_gap", F.col("stock_level") - F.col("reorder_point")) \
    .withColumn("at_risk",
        F.when(F.col("stock_level") <= F.col("reorder_point"), True).otherwise(False)) \
    .withColumn("days_to_stockout",
        F.when(
            F.col("daily_usage_rate") > 0,
            F.round(F.col("stock_level") / F.col("daily_usage_rate"), 1)
        ).otherwise(None)) \
    .orderBy("stock_gap")

print("Inventory risk — products at or below reorder point:")
inventory_risk.filter(F.col("at_risk") == True).show(20, truncate=False)

inventory_risk.write \
    .mode("overwrite") \
    .parquet(ANALYTICS["inventory_risk"])
print("Written: inventory_risk")

### Analytics 4: Supplier Scorecard

**Business question:** Which suppliers are underperforming on delivery reliability and quality? This scorecard ranks suppliers and flags those with defect rates or late delivery rates outside acceptable thresholds.

In [ ]:
supplier_scorecard = supplier.groupBy(
    "supplier_id", "supplier_name", "country"
).agg(
    F.avg("on_time_delivery_rate").alias("avg_on_time_delivery_rate"),
    F.avg("supplier_defect_rate").alias("avg_supplier_defect_rate"),
    F.avg("avg_lead_time_days").alias("avg_lead_time_days"),
    F.count("report_month").alias("months_reported")
).withColumn(
    "delivery_flag",
    F.when(F.col("avg_on_time_delivery_rate") < 0.90, "BELOW_TARGET").otherwise("OK")
).withColumn(
    "defect_flag",
    F.when(F.col("avg_supplier_defect_rate") > 0.05, "HIGH_DEFECT").otherwise("OK")
).withColumn(
    "overall_risk",
    F.when(
        (F.col("delivery_flag") == "BELOW_TARGET") & (F.col("defect_flag") == "HIGH_DEFECT"),
        "HIGH"
    ).when(
        (F.col("delivery_flag") == "BELOW_TARGET") | (F.col("defect_flag") == "HIGH_DEFECT"),
        "MEDIUM"
    ).otherwise("LOW")
).orderBy(F.desc("avg_supplier_defect_rate"))

print("Supplier scorecard:")
supplier_scorecard.show(20, truncate=False)

supplier_scorecard.write \
    .mode("overwrite") \
    .parquet(ANALYTICS["supplier_scorecard"])
print("Written: supplier_scorecard")

### Analytics 5: Supplier Defect Rate vs. Production Defect Rate

**Business question:** Do high supplier defect rates predict high production defect rates? This cross-domain join is the core insight connecting upstream supplier quality to downstream manufacturing outcomes.

In [ ]:
supplier_vs_production_defects = production_supplier.groupBy(
    "supplier_id", "supplier_name", "country"
).agg(
    F.avg("supplier_defect_rate").alias("avg_supplier_defect_rate"),
    F.avg("on_time_delivery_rate").alias("avg_on_time_delivery_rate"),
    F.sum("units_produced").alias("total_units_produced"),
    F.sum("units_defective").alias("total_units_defective"),
    F.round(
        F.sum("units_defective") / F.sum("units_produced") * 100, 2
    ).alias("production_defect_rate_pct")
).orderBy(F.desc("production_defect_rate_pct"))

print("Supplier defect rate vs. production defect rate:")
supplier_vs_production_defects.show(20, truncate=False)

supplier_vs_production_defects.write \
    .mode("overwrite") \
    .parquet(f"{ANALYTICS['supplier_scorecard']}/supplier_vs_production_defects")
print("Written: supplier_vs_production_defects")

### Analytics 6: Equipment Sensor Anomalies Correlated with Downtime

**Business question:** Do days with abnormally high temperature or vibration readings correspond to higher downtime and more defects? This supports predictive maintenance planning.

In [ ]:
# Define anomaly thresholds (adjust based on dataset distribution)
TEMP_THRESHOLD      = 200.0   # degrees C
VIBRATION_THRESHOLD = 15.0    # mm/s

sensor_anomaly_impact = sensors_production \
    .withColumn("temp_anomaly",
        F.when(F.col("max_temp_c") > TEMP_THRESHOLD, True).otherwise(False)) \
    .withColumn("vibration_anomaly",
        F.when(F.col("max_vibration") > VIBRATION_THRESHOLD, True).otherwise(False)) \
    .withColumn("any_anomaly",
        (F.col("temp_anomaly") | F.col("vibration_anomaly"))) \
    .groupBy("any_anomaly").agg(
        F.count("*").alias("shift_count"),
        F.avg("downtime_minutes").alias("avg_downtime_minutes"),
        F.avg(
            F.col("units_defective") / F.col("units_produced") * 100
        ).alias("avg_defect_rate_pct")
    ).orderBy("any_anomaly")

print("Sensor anomaly vs downtime/defect rate:")
sensor_anomaly_impact.show(truncate=False)

# Also write the full enriched dataset for deeper exploration
sensors_production.write \
    .mode("overwrite") \
    .parquet(ANALYTICS["equipment_monitoring"])
print("Written: equipment_monitoring")

## 9. Idempotency Check

All writes above use `.mode("overwrite")`, which means re-running this notebook produces the same outputs without creating duplicates. This satisfies the project's idempotency requirement.

The cleaning steps also include `.dropDuplicates()` on each dataset before writing to the curated zone, ensuring that even if the landing zone contains repeated rows, the curated and analytics outputs remain consistent.

## 10. Analytics Output Verification

In [ ]:
print("=== Analytics Zone Verification ===")

checks = [
    ("defect_by_line_shift",            f"{ANALYTICS['defect_analysis']}/defect_by_line_shift"),
    ("defect_code_summary",             f"{ANALYTICS['defect_analysis']}/defect_code_summary"),
    ("inventory_risk",                  ANALYTICS["inventory_risk"]),
    ("supplier_scorecard",              ANALYTICS["supplier_scorecard"]),
    ("supplier_vs_production_defects",  f"{ANALYTICS['supplier_scorecard']}/supplier_vs_production_defects"),
    ("equipment_monitoring",            ANALYTICS["equipment_monitoring"]),
]

for name, path in checks:
    df = spark.read.parquet(path)
    print(f"  {name}: {df.count():,} rows  |  columns: {df.columns}")

## 11. Stage 2 Completion Checklist

- Read all five raw datasets from the HDFS landing zone using PySpark
- Cleaned and standardized each dataset (nulls dropped, types cast, strings trimmed, values range-filtered)
- Deduplicated all datasets with `.dropDuplicates()`
- Wrote all five cleaned datasets to the curated zone as partitioned Parquet
- Joined production + quality inspections (Join 1: product_id)
- Joined production + supplier performance (Join 2: supplier_id)
- Joined sensor daily aggregates + production (Join 3: factory_id + date)
- Aggregated defect rate by factory, line, and shift (Analytics 1)
- Aggregated top defect codes by frequency and severity (Analytics 2)
- Identified inventory-at-risk products below reorder point (Analytics 3)
- Built supplier scorecard with delivery and defect risk flags (Analytics 4)
- Correlated supplier defect rate with production defect rate (Analytics 5)
- Correlated sensor anomalies with downtime and defect rate (Analytics 6)
- Wrote all analytics outputs to HDFS analytics zone as Parquet
- All writes use `mode("overwrite")` — pipeline is idempotent

In [ ]:
spark.stop()
print("SparkSession stopped.")